# RAG Evaluation Metrics

Runnable code patterns for evaluating Retrieval-Augmented Generation (RAG) systems.
No live API required — replace `simulate_judge()` with a real LLM call for production use.

**Prerequisites:** Read `0-README.md` in this folder before running.

| Section | Topic |
|---------|-------|
| §1 | RAG Triad: faithfulness (QAG), answer relevancy, context relevance |

## 1. RAG Metrics: Faithfulness & Answer Relevancy

For research agents that retrieve context before generating answers.
These metrics implement the RAG Triad described in `4-establishing-evaluation-framework.md`.

Three relationships to test independently:
- **Faithfulness** (groundedness) — is every claim in the answer supported by retrieved context?
- **Answer Relevancy** — does the answer actually address what the user asked?
- **Context Relevance** — were the retrieved chunks relevant to the query?

In [ ]:
# RAG evaluation without a live API call — pure code patterns
# In production, replace simulate_judge() with an actual LLM call.

import json

def simulate_judge(prompt: str) -> str:
    """Stub — replace with: anthropic_client.messages.create(...) or similar."""
    # Returns a deterministic fake response for demonstration
    if "supported" in prompt.lower() or "grounded" in prompt.lower():
        return '{"answer": "Yes"}'
    return '{"answer": "No"}'


# ── Faithfulness (QAG pattern) ────────────────────────────────────────────────
# Decompose the answer into individual claims, then verify each claim
# against the retrieved context with a yes/no question.
# Score = supported_claims / total_claims
# More reliable than asking "Is this answer faithful overall?" because it
# forces the judge to reason at the claim level, not the paragraph level.

def extract_claims(answer: str) -> list[str]:
    """
    In production: call an LLM to extract atomic factual claims.
    Here we split on sentences as a stand-in.
    """
    sentences = [s.strip() for s in answer.split(".") if s.strip()]
    return sentences

def check_claim_against_context(claim: str, context_chunks: list[str]) -> bool:
    """Ask the judge LLM whether the context supports this specific claim."""
    context_text = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(context_chunks))
    prompt = f"""Does the following context support the claim? Answer only Yes or No.

Claim: {claim}

Context:
{context_text}

Return JSON: {{"answer": "Yes"}} or {{"answer": "No"}}"""
    raw = simulate_judge(prompt)
    try:
        return json.loads(raw).get("answer", "No").strip().lower() == "yes"
    except json.JSONDecodeError:
        return False

def compute_faithfulness(answer: str, context_chunks: list[str]) -> dict:
    claims = extract_claims(answer)
    if not claims:
        return {"faithfulness": 0.0, "supported": 0, "total": 0, "claims": []}

    results = []
    for claim in claims:
        supported = check_claim_against_context(claim, context_chunks)
        results.append({"claim": claim, "supported": supported})

    supported_count = sum(1 for r in results if r["supported"])
    score = supported_count / len(claims)
    return {
        "faithfulness": round(score, 3),
        "supported": supported_count,
        "total": len(claims),
        "claims": results,
    }


# ── Answer Relevancy ──────────────────────────────────────────────────────────
# Measures whether the answer directly addresses what the user asked.
# Uses cosine similarity between the question embedding and back-generated
# questions from the answer. Here we use a lightweight keyword-overlap proxy.

def compute_answer_relevancy_simple(question: str, answer: str) -> dict:
    """
    Lightweight keyword-overlap proxy for answer relevancy.
    In production use: sentence-transformers cosine similarity, or
    DeepEval AnswerRelevancyMetric, or RAGAS answer_relevancy.
    """
    q_tokens = set(question.lower().split())
    a_tokens = set(answer.lower().split())
    # Remove stopwords inline
    stopwords = {"the", "a", "an", "is", "are", "was", "were", "of", "to", "in",
                 "for", "and", "or", "it", "its", "be", "by", "on", "at", "with"}
    q_keywords = q_tokens - stopwords
    a_keywords = a_tokens - stopwords

    if not q_keywords:
        return {"answer_relevancy": 0.0}

    overlap = len(q_keywords & a_keywords)
    score = overlap / len(q_keywords)
    return {
        "answer_relevancy": round(min(score, 1.0), 3),
        "q_keywords": sorted(q_keywords),
        "matched": sorted(q_keywords & a_keywords),
    }


# ── Context Relevance ─────────────────────────────────────────────────────────
# Fraction of retrieved chunks that actually contain content relevant to the query.

def compute_context_relevance(query: str, chunks: list[str]) -> dict:
    """
    Ask the judge whether each chunk is relevant to the query.
    Score = relevant_chunks / total_chunks
    Target: >= 0.7
    """
    if not chunks:
        return {"context_relevance": 0.0, "relevant": 0, "total": 0}

    relevant_count = 0
    chunk_results = []
    for chunk in chunks:
        prompt = f"""Is the following context chunk relevant to answering this query?
Query: {query}
Chunk: {chunk}
Return JSON: {{"answer": "Yes"}} or {{"answer": "No"}}"""
        raw = simulate_judge(prompt)
        try:
            is_relevant = json.loads(raw).get("answer", "No").strip().lower() == "yes"
        except json.JSONDecodeError:
            is_relevant = False
        if is_relevant:
            relevant_count += 1
        chunk_results.append({"chunk": chunk[:80] + "...", "relevant": is_relevant})

    score = relevant_count / len(chunks)
    return {
        "context_relevance": round(score, 3),
        "relevant": relevant_count,
        "total": len(chunks),
        "chunks": chunk_results,
    }


# ── Full RAG Triad Evaluation ─────────────────────────────────────────────────

def evaluate_rag_triad(
    query: str,
    answer: str,
    retrieved_chunks: list[str],
) -> dict:
    faithfulness   = compute_faithfulness(answer, retrieved_chunks)
    relevancy      = compute_answer_relevancy_simple(query, answer)
    ctx_relevance  = compute_context_relevance(query, retrieved_chunks)

    triad = {
        "faithfulness":      faithfulness["faithfulness"],
        "answer_relevancy":  relevancy["answer_relevancy"],
        "context_relevance": ctx_relevance["context_relevance"],
    }
    triad["rag_triad_avg"] = round(sum(triad.values()) / 3, 3)

    # Diagnosis
    if triad["context_relevance"] < 0.7:
        diagnosis = "RETRIEVER issue — wrong chunks fetched. Fix: embedding model, chunk size, top-k."
    elif triad["faithfulness"] < 0.8:
        diagnosis = "GENERATOR issue — hallucinating despite good context. Fix: prompt constraints, temperature."
    elif triad["answer_relevancy"] < 0.8:
        diagnosis = "ALIGNMENT issue — answer is factual but off-topic. Fix: system prompt, query reformulation."
    else:
        diagnosis = "System working correctly."

    return {**triad, "diagnosis": diagnosis, "_detail": {
        "faithfulness_detail": faithfulness,
        "relevancy_detail":    relevancy,
        "ctx_relevance_detail": ctx_relevance,
    }}


# ── Example ───────────────────────────────────────────────────────────────────

query = "What is SAP AI Core and how is it priced?"

retrieved_chunks = [
    "SAP AI Core is a managed AI service on SAP BTP that enables deployment and execution of AI functions.",
    "SAP AI Core offers three service plans: Free, Standard, and Extended.",
    "The BTP platform supports multiple runtimes including Cloud Foundry and Kyma.",  # less relevant
]

answer = (
    "SAP AI Core is a managed AI service on SAP BTP. "
    "It provides three pricing tiers: Free, Standard, and Extended. "
    "It also supports quantum computing."  # hallucinated claim
)

result = evaluate_rag_triad(query, answer, retrieved_chunks)

print("RAG Triad Scores")
print("─" * 40)
for k, v in result.items():
    if k != "_detail":
        print(f"  {k:<25} {v}")
print()
print("Faithfulness claim breakdown:")
for c in result["_detail"]["faithfulness_detail"]["claims"]:
    status = "✓" if c["supported"] else "✗"
    print(f"  {status}  {c['claim']}")